In [ ]:
# Настройка путей 
import sys
from pathlib import Path

# Получаем путь к корню проекта: поднимаемся из "Jupyter Notebooks/" на уровень выше
project_root = Path().resolve().parent

# Добавляем корень проекта в sys.path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Project root added to sys.path: {project_root}")

#### Импорт библиотек и загрузка данных

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from datetime import datetime
from config import RAW_DATA_DIR
from src.db.queries import run_query

# Загрузка данных
df = pd.read_csv(RAW_DATA_DIR / "third_wave_coffee_shop.csv", parse_dates=['datetime', 'sale_date'])


#### 2. Проверка качества данных (пропуски, дубликаты)

In [ ]:
# Пропущенные значения
print("Пропущенные значения:")
print(df.isnull().sum())

# Дубликаты (полные строки)
print(f"\nКоличество дубликатов: {df.duplicated().sum()}")

# Удаление дубликатов, если нужно (можно раскомментировать)
# df = df.drop_duplicates()

# Проверка уникальных значений в ключевых колонках
print("\nУникальные напитки:")
print(df['coffee_name'].value_counts().head(10))

#### 3. Преобразование дат (для временного анализа)

In [ ]:
# Добавление месяца и часа (если нужно для группировки)
df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour
df['date'] = df['datetime'].dt.date
display(df[['datetime', 'month', 'hour', 'date']].head())

In [ ]:
print(df.columns.tolist())

#### Структура данных:

- **Каждая строка = 1 напиток**

- **Каждая транзакция (`transaction_id`) = 1 заказ (чек)**

- `total_cost` — **одинаков** для всех строк одной транзакции

- **Все расчёты по чекам, выручке, времени, дням — только по уникальным `transaction_id`**

In [ ]:
# Создание df_tx — только уникальные транзакции
df_tx = df.drop_duplicates(subset='transaction_id').reset_index(drop=True)

# Добавляем количество напитков в заказе
df_tx['num_drinks'] = df.groupby('transaction_id').size().values

print(f"Уникальных транзакций: {len(df_tx)}")
print(f"Всего строк в df: {len(df)}")
print(f"Среднее количество напитков в заказе: {df_tx['num_drinks'].mean():.2f}")